# RescueVision - FloodNet train v2 (T4x2)

In [ ]:
import subprocess, sys, os, json, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = Path("/tmp/RescueVision")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/AliOmarAbdelhady/RescueVision.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

def mark(stage, ok=True, extra=""):
    """Progress ledger in /kaggle/working: survives kills where the log does not."""
    rec = {"t": time.strftime("%H:%M:%S"), "stage": stage, "ok": bool(ok), "extra": str(extra)[:1500]}
    with open(WORK / "_progress.txt", "a") as f:
        f.write(json.dumps(rec) + "\n")
    (WORK / "_status.json").write_text(json.dumps(rec, indent=1))
    print("[mark]", rec, flush=True)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
mark("setup:start")
# pin albumentations: newer versions (preinstalled on the GPU image) have a
# stricter shape check that rejects our verified-consistent (HxWx3, HxW) pairs;
# --upgrade forces the downgrade past pip's "already satisfied"; all transforms
# ALSO use is_check_shapes=False as a version-proof second layer
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "albumentations==2.0.8"], check=False)
import albumentations as _A
mark("setup:albumentations", extra=_A.__version__)
print("albumentations", _A.__version__)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch'], check=False)
import segmentation_models_pytorch as smp
print('smp', smp.__version__)
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# num_workers=0: DataLoader workers died on the Kaggle GPU image; pin_memory
# off too (useless with workers=0, and a hard-kill vector on some drivers)
NW = 0
PIN = False
print("torch", torch.__version__, "| device:", DEVICE)
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)
mark("setup:done", extra=str(DEVICE))

# per-epoch heartbeat: every trainer that logs an epoch also lands in the marks,
# so a silent kill still tells us how far training got and with what metrics
import rescuevision.training.logging_utils as _lu
_orig_log = _lu.CSVLogger.log
def _marked_log(self, metrics, epoch):
    try:
        mark(f"epoch{epoch}", extra=str({k: round(float(v), 4) for k, v in metrics.items()
                                         if isinstance(v, (int, float))})[:400])
    except Exception:
        pass
    return _orig_log(self, metrics, epoch)
_lu.CSVLogger.log = _marked_log
print("epoch heartbeat patch applied")

In [ ]:
import numpy as np
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from rescuevision.constants import IMAGENET_MEAN, IMAGENET_STD

def _nocheck(tf):
    try:
        return A.Compose(tf.transforms, is_check_shapes=False)
    except Exception:
        return tf
mark("patch:done")

In [ ]:
import time
import pandas as pd
from PIL import Image as _PIL

def _load_arr(path):
    path = Path(path)
    return np.load(path) if path.suffix == ".npy" else np.array(_PIL.open(path))

mark("tiles:wait")
TAR = None
for _attempt in range(80):   # 80 min: T4x2 pool can attach kernel sources slowly
    for _p in Path("/kaggle/input").rglob("floodnet_tiles_png.tar"):
        TAR = _p
        break
    if TAR is not None:
        break
    if _attempt % 4 == 0:
        _ents = [p.name for p in Path("/kaggle/input").iterdir()] if Path("/kaggle/input").exists() else []
        print(f"[{_attempt+1}/80] waiting, /kaggle/input: {_ents}", flush=True)
    time.sleep(60)
assert TAR is not None, "floodnet_tiles_png.tar never mounted"
_gib = TAR.stat().st_size / 1e9
print("tar:", TAR, f"{_gib:.2f} GB")
mark("tiles:mounted", extra=f"{_gib:.2f}GB")
subprocess.run(["tar", "-xf", str(TAR), "-C", "/tmp"], check=True)
TILES = Path("/tmp/floodnet_tiles")
_n_train = len(list((TILES / "train").glob("*_img.png")))
_n_val = len(list((TILES / "val").glob("*_img.png")))
print(f"tiles ready: train={_n_train} val={_n_val}")
mark("tiles:extracted", extra=f"train={_n_train} val={_n_val}")

In [ ]:
import logging
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from rescuevision.constants import NUM_FLOODNET_CLASSES
from rescuevision.models.losses import WeightedCEDiceLoss
from rescuevision.training.train_segmentation import train_segmentation, evaluate_segmentation
from rescuevision.utils.seed import set_seed
from rescuevision.data.transforms import get_segmentation_transforms
mark("floodnet:imports")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
OUTPUT = WORK
name = "floodnet_segformer_b2"
METRIC = "miou"
SEED = 42
AUG = {"horizontal_flip": 0.5, "vertical_flip": 0.5, "rotate90": 0.5, "brightness_contrast": 0.3, "normalize": True}
BS = 4
EPOCHS = 12
PATIENCE = 4

class FloodNetTileDataset(Dataset):
    def __init__(self, split_dir, transform=None):
        self.files = sorted(split_dir.glob("*_img.png"))
        self.transform = transform
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        base = str(self.files[i])[: -len("_img.png")]
        image = _load_arr(base + "_img.png")
        mask = _load_arr(base + "_ids.png")
        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug["image"], aug["mask"]
        mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
        return {"image": image, "mask": mask, "tile_id": Path(base).name}

set_seed(SEED)
train_ds = FloodNetTileDataset(TILES / "train", _nocheck(get_segmentation_transforms(AUG, "train", 512)))
val_ds = FloodNetTileDataset(TILES / "val", _nocheck(get_segmentation_transforms(AUG, "val", 512)))
print(f"train={len(train_ds)} val={len(val_ds)}")
mark("floodnet:data", extra=f"train={len(train_ds)}")
assert len(train_ds) > 0 and len(val_ds) > 0
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=NW, pin_memory=PIN, persistent_workers=bool(NW))
val_loader = DataLoader(val_ds, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=PIN)

counts = np.zeros(NUM_FLOODNET_CLASSES, dtype=np.int64)
for _f in (TILES / "train").glob("*_ids.png"):
    counts += np.bincount(_load_arr(_f).flatten(), minlength=NUM_FLOODNET_CLASSES)
print("class pixel counts:", counts.tolist())
assert counts.sum() > 0
freq = counts / counts.sum()
weights = 1.0 / np.sqrt(freq)
weights = weights / weights.sum() * NUM_FLOODNET_CLASSES
class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
print("class weights:", [round(w, 3) for w in class_weights.tolist()])
mark("floodnet:weights")

model = smp.Segformer(encoder_name="mit_b2", encoder_weights="imagenet",
                      in_channels=3, classes=NUM_FLOODNET_CLASSES).to(DEVICE)
mark("floodnet:model-ready")
loss_fn = WeightedCEDiceLoss(class_weights=class_weights, ce_weight=0.5, dice_weight=0.5, ignore_index=255)

try:
    _it = iter(train_loader)
    for _ in range(2):
        _b = next(_it)
        _img = _b["image"].to(DEVICE)
        _m = _b["mask"].to(DEVICE)
        with torch.amp.autocast(DEVICE.type, enabled=True):
            _lg = model(_img)
            _loss = loss_fn(_lg, _m)
        _loss.backward()
        model.zero_grad(set_to_none=True)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        mark("probe:ok", extra=f"peak={torch.cuda.max_memory_allocated()/1e9:.1f}GB")
    else:
        mark("probe:ok", extra="cpu")
except Exception:
    mark("probe:FAIL", extra=traceback.format_exc()[-1200:])
    raise

config = {"seed": SEED, "device": DEVICE.type,
          "training": {"epochs": EPOCHS, "lr": 6e-5, "weight_decay": 1e-4, "scheduler": "cosine",
                       "mixed_precision": True, "early_stopping_patience": PATIENCE,
                       "save_best_metric": METRIC, "gradient_accumulation_steps": 2}}
mark("floodnet:train-start", extra=f"train={len(train_ds)}")
best = train_segmentation(model, train_loader, val_loader, loss_fn, config,
                          output_dir=str(Path("/tmp/run") / name), is_binary=False,
                          num_classes=NUM_FLOODNET_CLASSES, ignore_index=255)
print("best:", best)
mark("floodnet:train-done", extra=str(best))

In [ ]:
import shutil as _shutil
_shutil.copytree(f"/tmp/run/{name}", WORK / name, dirs_exist_ok=True)
print("results copied to", WORK / name)
mark("copyout")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
log_df = pd.read_csv(next((WORK / name / "logs").glob("*.csv")))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log_df["epoch"], log_df["train_loss"], label="train")
if "val_loss" in log_df:
    axes[0].plot(log_df["epoch"], log_df["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(log_df["epoch"], log_df["miou"], color="tab:green")
axes[1].set_title(f"best miou={log_df['miou'].max():.4f}")
plt.tight_layout()
plt.savefig(WORK / name / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()
mark("curves:done")

In [ ]:
import json
ckpt = torch.load(WORK / name / "checkpoints" / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE).eval()
val_metrics = evaluate_segmentation(model, val_loader, DEVICE, is_binary=False,
                                    num_classes=NUM_FLOODNET_CLASSES, ignore_index=255)
print("VAL :", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in val_metrics.items()})
summary = {"run": name, "best_epoch": int(ckpt["epoch"]),
    f"best_val_{METRIC}": float(ckpt.get(f"best_{METRIC}")),
    "val": {k: round(v, 4) for k, v in val_metrics.items() if not isinstance(v, list)}}
with open(WORK / name / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
mark("floodnet:eval-done")